<a href="https://colab.research.google.com/github/maekuhi/GTU-CSE685RobotControlTheory/blob/main/CSE685HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course: Robot Control Theory
* Professor ZERGEROGLU
* HW1
* Student: Mahdi AMANI ESTALKHKUHI
----------------------------------


##step1: Define Parameters
OPW robot parameters (KUKA youBot Arm) from Table I – OPW Parameter Collection

##step2: Desired End-Effector Pose

The desired pose is defined by a position vector and a rotation matrix.

Position:

$$
\mathbf{u}_0 =
\begin{bmatrix}
u_{x0} \\
u_{y0} \\
u_{z0}
\end{bmatrix}
$$

Orientation:

$$
R_0^e
$$

##step3: First Joint Angle

The first joint angle is obtained from the projection of the target position in the XY-plane:

$$
\theta_1 =
\operatorname{atan2}(u_{y0}, u_{x0})
$$

##step4: Wrist Center

The wrist center is computed by subtracting the tool offset along the end-effector z-axis.

$$
\mathbf{C} =
\mathbf{u}_0 -
c_4 R_0^e
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

where

$$
\mathbf{C} =
\begin{bmatrix}
c_{x0} \\
c_{y0} \\
c_{z0}
\end{bmatrix}
$$

##step5: Compute $c_{x1}$

The wrist center is transformed into the coordinate system of joint 1.

$$
c_{x1} =
\sqrt{c_{x0}^2 + c_{y0}^2 - b^2}
$$

##step 6: Compute $n_{x1}$

The horizontal distance between the wrist center and the first arm offset is

$$
n_{x1} = c_{x1} - a_1
$$

##step7: Distance $s$

The distance between the second joint axis and the wrist center is

$$
s =
\sqrt{n_{x1}^2 + (c_{z0} - c_1)^2}
$$

##step8: Constant $k$

The geometric constant used in the triangle relation is

$$
k =
\sqrt{a_2^2 + c_3^2}
$$

#step9: Angle $\psi_2$

Using the cosine law:

$$
\psi_2 =
\cos^{-1}
\left(
\frac{s^2 + c_2^2 - k^2}{2 s c_2}
\right)
$$

##step10: Joint Angle $\theta_2$

The second joint angle is

$$
\theta_2 =
\operatorname{atan2}(n_{x1}, c_{z0} - c_1) - \psi_2
$$

##step 11: Joint Angle $\theta_3$

First compute

$$
\psi_3 =
\operatorname{atan2}(a_2, c_3)
$$

Then

$$
\theta_3 =
\cos^{-1}
\left(
\frac{s^2 - c_2^2 - k^2}{2 c_2 k}
\right)
-
\psi_3
$$

##step 12: Rotation Matrix $R_0^c$

The orientation of the wrist frame relative to the base frame is

$$
R_0^c =
\begin{bmatrix}
c_1c_2c_3 - c_1s_2s_3 & -s_1 & c_1c_2s_3 + c_1s_2c_3 \\
s_1c_2c_3 - s_1s_2s_3 & c_1 & s_1c_2s_3 + s_1s_2c_3 \\
-s_2c_3 - c_2s_3 & 0 & -s_2s_3 + c_2c_3
\end{bmatrix}
$$

where

$$
s_i = \sin(\theta_i), \qquad
c_i = \cos(\theta_i)
$$

##step 13: Wrist Orientation

The orientation of the wrist relative to the end-effector is

$$
R_c^e =
(R_0^c)^T R_0^e
$$

##step 14: Wrist Joint Angles

The last three joint angles are extracted from the rotation matrix.

Joint 5:

$$
\theta_5 =
\operatorname{atan2}
\left(
\sqrt{1 - R_{33}^2},
R_{33}
\right)
$$

Joint 4:

$$
\theta_4 =
\operatorname{atan2}(R_{23}, R_{13})
$$

Joint 6:

$$
\theta_6 =
\operatorname{atan2}(R_{32}, -R_{31})
$$

In [7]:

# Values converted from mm to meters
import numpy as np

# step1 OPW parameters (KUKA youBot)
# -----------------------------
a1 = 0.033
a2 = 0.0
b  = 0.0
c1 = 0.147
c2 = 0.155
c3 = 0.135
c4 = 0.2175

# step2 Desired end-effector pose
# -----------------------------
u0 = np.array([0.25, 0.05, 0.20])   # position
R0e = np.eye(3)                     # orientation

# Step3 θ1
# -----------------------------
theta1 = np.arctan2(u0[1], u0[0])

# Step4 Wrist center
# -----------------------------
z_axis = np.array([0,0,1])
C = u0 - c4 * (R0e @ z_axis)

cx, cy, cz = C

# Step5 — cx1
# -----------------------------
cx1 = np.sqrt(cx**2 + cy**2 - b**2)

# Step6 — nx1
# -----------------------------
nx1 = cx1 - a1

# Step7 — distance s
# -----------------------------
s = np.sqrt(nx1**2 + (cz - c1)**2)

# Step8 — constant k
# -----------------------------
k = np.sqrt(a2**2 + c3**2)

# Step9 — ψ2
# -----------------------------
cos_psi2 = (s**2 + c2**2 - k**2) / (2*s*c2)
cos_psi2 = np.clip(cos_psi2, -1.0, 1.0)

psi2 = np.arccos(cos_psi2)

# Step10 — θ2
# -----------------------------
theta2 = np.arctan2(nx1, cz - c1) - psi2

# Step11 — θ3
# -----------------------------
psi3 = np.arctan2(a2, c3)

cos_theta3 = (s**2 - c2**2 - k**2) / (2*c2*k)
cos_theta3 = np.clip(cos_theta3, -1.0, 1.0)

theta3 = np.arccos(cos_theta3) - psi3

# Step12 — R0c
# -----------------------------
s1, c1t = np.sin(theta1), np.cos(theta1)
s2, c2t = np.sin(theta2), np.cos(theta2)
s3, c3t = np.sin(theta3), np.cos(theta3)

R0c = np.array([
[c1t*c2t*c3t - c1t*s2*s3, -s1, c1t*c2t*s3 + c1t*s2*c3t],
[s1*c2t*c3t - s1*s2*s3,  c1t, s1*c2t*s3 + s1*s2*c3t],
[-s2*c3t - c2t*s3, 0, -s2*s3 + c2t*c3t]
])

# Step13 — Wrist orientation
# -----------------------------
Rce = R0c.T @ R0e

# Step14 — θ4 θ5 θ6
# -----------------------------
theta5 = np.arctan2(np.sqrt(1 - Rce[2,2]**2), Rce[2,2])
theta4 = np.arctan2(Rce[1,2], Rce[0,2])
theta6 = np.arctan2(Rce[2,1], -Rce[2,0])

# -----------------------------
# Final joint angles
# -----------------------------
angles = np.array([theta1, theta2, theta3, theta4, theta5, theta6])

print("Joint angles (radians):")
print(angles)

Joint angles (radians):
[0.19739556 1.92092214 0.61951982 3.14159265 2.54044197 2.94419709]
